In [2]:
import numpy as np
from datetime import datetime
import smtplib
from selenium import webdriver
import os
import pandas as pd
#For Prediction
from sklearn.linear_model import LinearRegression
from sklearn import preprocessing#,cross_validation this is no longer aviliable
from sklearn.model_selection import train_test_split #use this instead
#For Stock Data
from iexfinance.stocks import get_historical_data
import pdb#for debug
from iexfinance.refdata import get_symbols
import matplotlib.pyplot as plt
import copy


In [3]:
#Download Historical stock data

start = datetime(2019, 10, 2)
end = datetime(2019, 10, 15)

df = get_historical_data("TSLA", start, end, output_format='pandas', token="pk_422a359c341b427ea05864740c233fe3")
df.to_csv('TSLA.csv') 
data = pd.read_csv('TSLA.csv')

In [4]:
#Model building and testing
def sendMessage(text):
    # If you're using Gmail to send the message, you might need to 
    # go into the security settings of your email account and 
    # enable the "Allow less secure apps" option 
    username = "shudizhao10310@gmail.com"
    password = "shudizhao923"

    to = "shudizhao923@gmail.com"
    message = text

    Subject = "TSLA Stock Prediction"
    msg = 'Subject: {}\n\n{}'.format(Subject, text)
    
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(username, password)
    server.sendmail(username, to, msg)
    server.quit()

    print('Sent')


data = pd.read_csv('TSLA.csv')
df = data[['close', 'high', 'low', 'open','volume', 'change']]
df['prediction'] = df['close'].shift(-1)
df.dropna(inplace=True)


#Predicting the Stock price in the future
X = np.array(df.drop(['prediction'], 1))
Y = np.array(df['prediction'])
X = preprocessing.scale(X)
X_prediction = X[-1:]
Y_ans = Y[-1:]
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

#Performing the Regression on the training data
clf = LinearRegression()
clf.fit(X_train, Y_train)
prediction = (clf.predict(X_prediction))
result = clf.score(X_train, Y_train)

output = ("\n\nStock: " + 'TSLA' + "\nClose Price on " + str(data.loc[data.index[[-2]], 'Unnamed: 0'].item()) + ": $" + str(data.loc[data.index[[-2]].item(), 'close']) + "\nPrediction for tomorrow closing: $%.2f" %
    (prediction[0]) + "\nActuall closing: $%.2f" % (Y_ans[0]) + "\nModel Accuracy: %.2f%%" % (result*100.0))

sendMessage(output)

/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:45: FutureWarning: `item` has been deprecated and will be removed in a future version


Sent


In [5]:
# Final application 
def getStocks():
    #Navigating to the Yahoo stock screener
    
    
    driver = webdriver.Chrome(
        '/Users/shudizhao/Desktop/Directed Study/chromedriver')
    url = "https://finance.yahoo.com/screener/predefined/aggressive_small_caps?offset=0&count=202"
    driver.get(url)

    #Creating a stock list and iterating through the ticker names on the stock screener list
    #Xpath: //*[@id="scr-res-table"]/div[1]/table/tbody/tr[1]/td[1]/a
    ticker = driver.find_element_by_xpath(
        '//*[@id = "scr-res-table"]/div[1]/table/tbody/tr[1]/td[1]/a')

    #stock = ticker.text
    stock = copy.deepcopy(ticker.text)
    print(stock)
    driver.quit()
    
    return predictData(stock)
    

        
def sendMessage(text):
    # If you're using Gmail to send the message, you might need to 
    # go into the security settings of your email account and 
    # enable the "Allow less secure apps" option 
    username = "shudizhao1031@gmail.com"
    password = "shudizhao923"
    
    to = "shudizhao923@gmail.com"
    message = text

    Subject = "Stock Prediction"
    msg = 'Subject: {}\n\n{}'.format(Subject, text)
    
    server = smtplib.SMTP('smtp.gmail.com', 587)
    server.starttls()
    server.login(username, password)
    server.sendmail(username, to, msg)
    server.quit()

    print('Sent')
    
def predictData(stock):
    
    start = datetime(2018, 10, 2)
    end = datetime(2019, 10, 2)
    
    df = get_historical_data(stock, start, end, output_format='pandas', token="pk_422a359c341b427ea05864740c233fe3") 
    csv_name = ( stock + '.csv')
    df.to_csv(csv_name)
    
    data = pd.read_csv(csv_name)
    df = data[['close', 'high', 'low', 'open','volume', 'change']]
    df['prediction'] = df['close'].shift(-1)
    df.dropna(inplace=True)

    #Predicting the Stock price in the future
    X = np.array(df.drop(['prediction'], 1))
    Y = np.array(df['prediction'])
    X = preprocessing.scale(X)
    X_prediction = X[-1:]
    Y_ans = Y[-1:]
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

    #Performing the Regression on the training data
    clf = LinearRegression()
    clf.fit(X_train, Y_train)
    prediction = (clf.predict(X_prediction))
    result = clf.score(X_train, Y_train)

    output = ("\n\nStock: " + str(stock) + "\nClose Price on " + str(data.loc[data.index[[-2]], 'Unnamed: 0'].item()) + ": $" + str(data.loc[data.index[[-2]].item(), 'close']) + "\nPrediction for tomorrow closing: $%.2f" %
    (prediction[0]) + "\nActuall closing: $%.2f" % (Y_ans[0]) + "\nModel Accuracy: %.2f%%" % (result*100.0))

    sendMessage(output)
    
if __name__ == '__main__':
    getStocks()

OGI


/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
/Applications/Jupter/anaconda3/lib/python3.7/site-packages/ipykernel_launcher.py:75: FutureWarning: `item` has been deprecated and will be removed in a future version


Sent
